# Evolution of the tied run

Same animation as folder 18, on the weight-tied run from `train.py`.

**What tying does to the picture.** The tie is enforced on the full matrices — a gradient hook
replaces each group's gradient by its sum, and `retie()` re-averages after every step — so the
stored entries are already the single parameter *broadcast* across its group. Nothing has to be
expanded here: every cell of a group holds the same number and they move together, frame by
frame.

| block | groups | cells → dof |
|---|---|---|
| L1 $W_{qk}$[0:8, 0:8] | four 4×4 quadrants | 64 → 4 |
| L2 $W_{qk}$[0:8, 0:8] (A) | left 8×4, right 8×4 | 64 → 2 |
| L2 $W_{qk}$[0:4, 108:112] (B) | diagonal, off-diagonal | 16 → 2 |
| L2 $W_{qk}$[108:112, 108:112] (C) | diagonal, off-diagonal | 16 → 2 |

L1's 100×100 position block is untied — every entry moves on its own.

Group boundaries are drawn in **black** over each tied panel, so you can see which cells are
one parameter. Values are the **raw entries of $W_{qk}$** — no centering, no rescaling.

In [7]:
import os, json, glob, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import HTML, display
plt.rcParams['animation.embed_limit'] = 300

V, Vs, T = 8, 4, 100
D1, D2 = V + T, 2 * (V + T)          # 108, 216

def discover_run():
    '''the run with the most epochs that has snapshots -- falls back to the newest'''
    cands = [d for d in glob.glob('runs/*/') if glob.glob(os.path.join(d, 'snapmeta_*.json'))]
    if not cands:
        raise FileNotFoundError('no run in runs/ has snapshots yet -- start train.py first')
    def eps(d):
        m = re.search(r'_ep-(\d+)', d)
        return int(m.group(1)) if m else -1
    return max(cands, key=lambda d: (eps(d), os.path.getmtime(d))).rstrip('/')

RUN = discover_run()            # or set it yourself, e.g. RUN = 'runs/tied_..._ep-500'
print(RUN, '\n')
for _t in (0.6, 0.8, 1.0):
    _f = f'{RUN}/snapmeta_x_{_t}.json'
    if not os.path.exists(_f):
        print(f'  alpha={_t}: not started'); continue
    _m = json.load(open(_f))
    _done = _m['w_rows_written'] >= _m['rows_allocated']
    print(f"  alpha={_t}: w={_m['w_rows_written']}/{_m['rows_allocated']}  g={_m['g_rows_written']}  "
          f"{'complete' if _done else 'STILL TRAINING -- cells below use what exists'}")

def aligned_losses(thr):
    '''train and test loss indexed by the weights they describe.

    train.py already evaluates BOTH after the step, so no shift is needed here -- unlike
    folder 18, where train_loss was the pre-step value and lagged test by one. `train_loss_pre`
    is kept in the metrics for exactly that comparison.
    '''
    h = json.load(open(f'{RUN}/metrics_seed0_x_{thr}.json'))['history']
    return (np.asarray(h['step']), np.asarray(h['train_loss']), np.asarray(h['test_loss']),
            h['test_loss'][-1], h['step'][-1])

def group_map(meta, param, mask):
    '''full-shape int array: which tie group each cell belongs to (-1 = untied)'''
    gid = np.full(meta['shapes'][param], -1, dtype=int)
    pos = np.argwhere(mask)                       # i-th True cell, row-major -- the row layout
    for k, (_, idxs) in enumerate(sorted(meta.get('tie_groups', {}).get(param, {}).items())):
        for i in idxs:
            gid[tuple(pos[i])] = k
    return gid

def panel(ax, A, title, vmax, annot, gid=None):
    v = vmax if vmax > 0 else 1.0
    ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
              norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    if annot:
        lab = [['0' if z == 0 else (f'{z:.3g}' if abs(z) >= 0.01 else f'{z:.1e}') for z in row]
               for row in A]
        wid = max(len(x) for row in lab for x in row)
        fs = (12.5 if A.shape[0] <= Vs else 6.4) * min(1.0, 5.0 / wid)
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, lab[i][j], ha='center', va='center', fontsize=fs,
                        color='white' if abs(A[i, j]) > v / 3 else 'black')
        if A.shape[0] <= Vs:
            ax.set_xticks(range(A.shape[1])); ax.set_yticks(range(A.shape[0]))
            ax.set_xticklabels([f'k{q}' for q in range(A.shape[1])], fontsize=8)
            ax.set_yticklabels([f'q{q}' for q in range(A.shape[0])], fontsize=8)
            ax.tick_params(length=0)
        else:
            ax.set_xticks([]); ax.set_yticks([])
    else:
        ax.set_xticks([]); ax.set_yticks([])
    if gid is not None:                       # outline the tie groups: one parameter each
        for i in range(gid.shape[0]):
            for j in range(gid.shape[1]):
                if j + 1 < gid.shape[1] and gid[i, j] != gid[i, j + 1]:
                    ax.plot([j + .5, j + .5], [i - .5, i + .5], color='k', lw=2.2)
                if i + 1 < gid.shape[0] and gid[i, j] != gid[i + 1, j]:
                    ax.plot([j - .5, j + .5], [i + .5, i + .5], color='k', lw=2.2)
    ax.set_title(f'{title}\n$\\|\\cdot\\|_F$ {np.linalg.norm(A):.4g}', fontsize=9)

runs/tied_opt-adam_lr-1.0_init-zeros_wov-fixed_blocks-relevant_readout-bias_ep-500 

  alpha=0.6: w=501/501  g=500  complete
  alpha=0.8: not started
  alpha=1.0: not started


## The animation

`ANIM_FROM` / `ANIM_TO` are epochs, both inclusive. Colour scales are fixed across the frames
*shown*, so a late window is scaled to itself — which is what makes small late changes visible,
and also means two windows are not comparable by colour.

In [ ]:
ANIM_THR  = 0.6      # 0.6, 0.8 or 1.0
ANIM_FROM = 0        # first epoch  (inclusive)
ANIM_TO   =  50    # last  epoch  (inclusive)
LOSS_ROW_H = 1.24     # height of the loss row, in units of one matrix row (0.62 = old)
ANIM_DPI   = 88       # six panels wide now -- drop this if the notebook feels heavy

# A 10x10 window from the MIDDLE of layer 1's 100x100 position block, blown up to a panel
# of its own and placed third, next to the full block. The full 100x100 is 9,900 near-identical
# ramp rows in which nothing is legible; this window is where the ramp actually lives -- it
# straddles the causal diagonal, so the decay with distance is visible cell by cell.
ZOOM_AT, ZOOM_N = 45, 10                      # position block rows/cols [45:55]
ZOOM = slice(V + ZOOM_AT, V + ZOOM_AT + ZOOM_N)

# the six panels: (param, row slice, col slice, label, annotate?)
CUT = [('layer1.W_qk', slice(0, V),        slice(0, V),        'L1 tok x tok',     True),
       ('layer1.W_qk', slice(V, D1),       slice(V, D1),       'L1 pos x pos',     False),
       ('layer1.W_qk', ZOOM,               ZOOM,
        f'L1 pos x pos  [{ZOOM_AT}:{ZOOM_AT + ZOOM_N}]  zoom',                     True),
       ('layer2.W_qk', slice(0, V),        slice(0, V),        'L2 A  noise suppr.', True),
       ('layer2.W_qk', slice(0, Vs),       slice(D1, D1 + Vs), 'L2 B  induction',  True),
       ('layer2.W_qk', slice(D1, D1 + Vs), slice(D1, D1 + Vs), 'L2 C  state',      True)]

def animate(thr, first=0, last=50, interval=450):
    mp = f'{RUN}/snapmeta_x_{thr}.json'
    if not os.path.exists(mp):
        raise FileNotFoundError(f'no snapshots for alpha={thr} in {RUN}')
    mt = json.load(open(mp))
    kmax = min(mt['w_rows_written'] - 1, mt['g_rows_written'] - 1)   # step k needs w[k] AND g[k]
    a, b = max(0, int(first)), min(int(last), kmax)
    if b < a:
        raise ValueError(f'empty range [{first}, {last}] -> [{a}, {b}]; '
                         f'this run has steps 0..{kmax} for alpha={thr}')
    if b < int(last):
        print(f'alpha={thr}: only steps 0..{kmax} recorded so far -> showing [{a}, {b}]')
    n = b - a + 1
    print(f'alpha={thr}: epochs {a}..{b}   ({n} frames)')

    Mk = {p: np.load(f'{RUN}/mask_{p}_x_{thr}.npy') for p in mt['params']}
    GID = {p: group_map(mt, p, Mk[p]) for p in mt['params']}
    Wf = {p: np.asarray(np.load(f'{RUN}/w_{p}_x_{thr}.npy', mmap_mode='r')[a:b + 1])
          for p in mt['params']}
    Gf = {p: np.asarray(np.load(f'{RUN}/g_{p}_x_{thr}.npy', mmap_mode='r')[a:b + 1])
          for p in mt['params']}

    def blocks(src, p, rs, cs):
        '''raw entries -- already the tied parameter broadcast over its group'''
        out = np.empty((n, rs.stop - rs.start, cs.stop - cs.start))
        for k in range(n):
            A = np.zeros(mt['shapes'][p]); A[Mk[p]] = src[p][k]; out[k] = A[rs, cs]
        return out

    Ws = [blocks(Wf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    Gs = [blocks(Gf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    vw = [float(np.abs(x).max()) for x in Ws]
    vg = [float(np.abs(x).max()) for x in Gs]
    gids = [GID[p][rs, cs] if an else None for p, rs, cs, _, an in CUT]

    st, tr, te, fin, nlast = aligned_losses(thr)
    st, tr, te = st[a:b + 1], tr[a:b + 1], te[a:b + 1]

    nb_ = len(CUT)
    # Three rows. The loss curve is no longer a narrow column on the right: it gets its own
    # row underneath both matrix rows, stretched across the full width, with the same panel
    # height it had before (LOSS_ROW_H is in units of one matrix row).
    f = plt.figure(figsize=(4.6 * nb_, 9.2 + 4.6 * LOSS_ROW_H + 1.55), dpi=ANIM_DPI)
    g = f.add_gridspec(3, nb_, height_ratios=[1, 1, LOSS_ROW_H], wspace=0.10, hspace=0.16)
    ax = np.array([[f.add_subplot(g[r, c_]) for c_ in range(nb_)] for r in range(2)])
    al = f.add_subplot(g[2, :])
    f.subplots_adjust(top=0.92, bottom=0.06, left=0.025, right=0.99)

    def dr(i_):
        ep = a + i_
        for a_ in ax.ravel():
            a_.clear()
        for j_, (_, _, _, ttl, an) in enumerate(CUT):
            panel(ax[0][j_], Ws[j_][i_], f'W  {ttl}', vw[j_], an, gids[j_])
            panel(ax[1][j_], Gs[j_][i_], f'grad  {ttl}', vg[j_], an, gids[j_])
        al.clear()
        al.plot(st, tr, color='#1f77b4', lw=1.8, label='train')
        al.plot(st, te, color='#ff7f0e', lw=1.4, ls='--', label='test')
        al.axvline(ep, color='k', lw=1.1, alpha=0.65)
        al.plot([ep], [tr[i_]], 'o', color='#1f77b4', ms=8, zorder=5)
        al.plot([ep], [te[i_]], 'o', color='#ff7f0e', ms=8, zorder=5)
        al.annotate(f'epoch {ep}\ntrain {tr[i_]:.5f}\ntest  {te[i_]:.5f}',
                    xy=(0.96, 0.95), xycoords='axes fraction', ha='right', va='top',
                    fontsize=8.5, family='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='0.7', alpha=0.9))
        al.set_xlim(a - 0.5, b + 0.5)
        al.set_xlabel('optimizer step (= epoch)', fontsize=9)
        al.set_ylabel('cross entropy', fontsize=9); al.tick_params(labelsize=8)
        al.set_title(f'loss over epochs {a}..{b}\n(run ends at {fin:.5f} after {nlast})',
                     fontsize=9)
        al.grid(alpha=0.3); al.legend(fontsize=8, loc='lower left')
        note = '   --  zero init: W all zero, and layer 1 has NO gradient yet' if ep == 0 else ''
        f.suptitle(f'$\\alpha$={thr}   epoch {ep}   (range {a}..{b})   '
                   f'top: $W$ before update {ep+1};  middle: $g[{ep}]$ at those weights;  '
                   f'bottom: the loss'
                   f'   [black outlines = one tied parameter]{note}', fontsize=13)
        return []

    an_ = animation.FuncAnimation(f, dr, frames=n, interval=interval, blit=False)
    plt.close(f)
    html = an_.to_jshtml()
    print(f'  animation {len(html) / 1e6:.1f} MB   ({nb_} panels x 2 rows + loss row)')
    return HTML(html)

display(animate(ANIM_THR, ANIM_FROM, ANIM_TO))